In [1]:
import json, math
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_file
from tokenizers import Tokenizer

In [2]:
# FIX #1: partial_rotary_factor support — Moonshine only rotates 90% of head dims
def apply_rope(q, k, seq_len, head_dim, partial_rotary_factor=0.9, theta=10000.0):
    # q,k: [B, H, T, head_dim]
    rotary_dim = int(head_dim * partial_rotary_factor)  # 32 out of 36 for moonshine-tiny
    # Make rotary_dim even
    rotary_dim = rotary_dim - (rotary_dim % 2)
    
    inv_freq = 1.0 / (theta ** (torch.arange(0, rotary_dim, 2, device=q.device).float() / rotary_dim))
    t = torch.arange(seq_len, device=q.device).float()
    freqs = torch.einsum('i,j->ij', t, inv_freq)  # [T, rotary_dim//2]
    emb = torch.cat([freqs, freqs], dim=-1)  # [T, rotary_dim]
    cos, sin = emb.cos(), emb.sin()
    
    # Match transformers implementation: repeat_interleave for cos/sin
    cos = cos[..., : cos.shape[-1] // 2].repeat_interleave(2, dim=-1)
    sin = sin[..., : sin.shape[-1] // 2].repeat_interleave(2, dim=-1)
    
    def rotate_half(x):
        x1 = x[..., 0::2]
        x2 = x[..., 1::2]
        return torch.stack((-x2, x1), dim=-1).flatten(-2)
    
    # Split into rotary and passthrough parts
    q_rot, q_pass = q[..., :rotary_dim], q[..., rotary_dim:]
    k_rot, k_pass = k[..., :rotary_dim], k[..., rotary_dim:]
    
    q_embed = (q_rot * cos) + (rotate_half(q_rot) * sin)
    k_embed = (k_rot * cos) + (rotate_half(k_rot) * sin)
    
    q_out = torch.cat([q_embed, q_pass], dim=-1)
    k_out = torch.cat([k_embed, k_pass], dim=-1)
    return q_out, k_out

In [3]:
# FIX #2: SwiGLU must have bias=True (default) to match the HF checkpoint
class SwiGLU(nn.Module):
    def __init__(self, dim, hidden):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden * 2)   # bias=True to match checkpoint
        self.fc2 = nn.Linear(hidden, dim)        # bias=True to match checkpoint
    def forward(self, x):
        x = self.fc1(x)
        x, gate = x.chunk(2, dim=-1)
        return self.fc2(F.silu(gate) * x)

In [4]:
class Attention(nn.Module):
    def __init__(self, dim, heads, num_kv_heads=None):
        super().__init__()
        self.heads = heads
        self.num_kv_heads = num_kv_heads if num_kv_heads is not None else heads
        self.dh = dim // heads
        self.num_kv_groups = heads // self.num_kv_heads
        
        # attention_bias=False in config, so bias=False for all projections
        self.q_proj = nn.Linear(dim, heads * self.dh, bias=False)
        self.k_proj = nn.Linear(dim, self.num_kv_heads * self.dh, bias=False)
        self.v_proj = nn.Linear(dim, self.num_kv_heads * self.dh, bias=False)
        self.o_proj = nn.Linear(dim, dim, bias=False)  # renamed from 'out' to 'o_proj' to match checkpoint
        
    def repeat_kv(self, x, n_rep):
        B, n_kv_heads, T, D = x.shape
        if n_rep == 1:
            return x
        return x[:, :, None, :, :].expand(B, n_kv_heads, n_rep, T, D).reshape(B, n_kv_heads * n_rep, T, D)
    
    def forward(self, x, mask=None, key_value_states=None):
        B,T,C = x.shape
        
        q = self.q_proj(x).view(B, T, self.heads, self.dh).transpose(1, 2)
        
        if key_value_states is not None:
            # Cross-attention: use encoder states for K/V
            k = self.k_proj(key_value_states).view(B, -1, self.num_kv_heads, self.dh).transpose(1, 2)
            v = self.v_proj(key_value_states).view(B, -1, self.num_kv_heads, self.dh).transpose(1, 2)
        else:
            # Self-attention
            k = self.k_proj(x).view(B, T, self.num_kv_heads, self.dh).transpose(1, 2)
            v = self.v_proj(x).view(B, T, self.num_kv_heads, self.dh).transpose(1, 2)
        
        # Apply RoPE only for self-attention (not cross-attention)
        if key_value_states is None:
            q, k = apply_rope(q, k, T, self.dh)
        
        # Repeat K/V if using GQA
        k = self.repeat_kv(k, self.num_kv_groups)
        v = self.repeat_kv(v, self.num_kv_groups)
        
        attn = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
        out = attn.transpose(1, 2).reshape(B, T, C)
        return self.o_proj(out)

In [5]:
class EncoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.self_attn = Attention(cfg['hidden_size'], cfg['encoder_num_attention_heads'], cfg.get('encoder_num_key_value_heads'))
        # FIX #3: Use named submodules that match HF checkpoint keys
        self.mlp = nn.ModuleDict({
            'fc1': nn.Linear(cfg['hidden_size'], cfg['intermediate_size']),
            'fc2': nn.Linear(cfg['intermediate_size'], cfg['hidden_size'])
        })
        # FIX #4: LayerNorm bias=False to match HF checkpoint (no bias weights in checkpoint)
        self.input_layernorm = nn.LayerNorm(cfg['hidden_size'], bias=False)
        self.post_attention_layernorm = nn.LayerNorm(cfg['hidden_size'], bias=False)
    def forward(self, x, mask=None):
        x = x + self.self_attn(self.input_layernorm(x), mask)
        # Encoder uses GELU activation (encoder_hidden_act = "gelu")
        h = F.gelu(self.mlp['fc1'](self.post_attention_layernorm(x)))
        x = x + self.mlp['fc2'](h)
        return x

class DecoderLayer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # FIX #5: Config uses 'decoder_num_attention_heads', NOT 'num_attention_heads'
        num_heads = cfg['decoder_num_attention_heads']
        num_kv_heads = cfg.get('decoder_num_key_value_heads', num_heads)
        
        self.self_attn = Attention(cfg['hidden_size'], num_heads, num_kv_heads)
        self.encoder_attn = Attention(cfg['hidden_size'], num_heads, num_kv_heads)  # renamed from cross_attn
        self.mlp = SwiGLU(cfg['hidden_size'], cfg['intermediate_size'])
        # FIX #4: LayerNorm bias=False
        self.input_layernorm = nn.LayerNorm(cfg['hidden_size'], bias=False)
        self.post_attention_layernorm = nn.LayerNorm(cfg['hidden_size'], bias=False)
        self.final_layernorm = nn.LayerNorm(cfg['hidden_size'], bias=False)  # renamed from ln3
    def forward(self, x, enc, self_mask=None, cross_mask=None):
        x = x + self.self_attn(self.input_layernorm(x), self_mask)
        x = x + self.encoder_attn(self.post_attention_layernorm(x), cross_mask, key_value_states=enc)
        x = x + self.mlp(self.final_layernorm(x))
        return x

# ---------- Full model ----------
class Moonshine(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        # Use nn.ModuleDict so parameter names match HF checkpoint hierarchy:
        # HF key: model.encoder.conv1.weight → our key: encoder.conv1.weight
        self.encoder = nn.ModuleDict({
            'conv1': nn.Conv1d(1, cfg['hidden_size'], kernel_size=127, stride=64, bias=False),
            'conv2': nn.Conv1d(cfg['hidden_size'], 2 * cfg['hidden_size'], kernel_size=7, stride=3),
            'conv3': nn.Conv1d(2 * cfg['hidden_size'], cfg['hidden_size'], kernel_size=3, stride=2),
            'groupnorm': nn.GroupNorm(num_groups=1, num_channels=cfg['hidden_size'], eps=1e-5),
            'layers': nn.ModuleList([EncoderLayer(cfg) for _ in range(cfg['encoder_num_hidden_layers'])]),
            'layer_norm': nn.LayerNorm(cfg['hidden_size'], bias=False),
        })
        dec_layers = cfg['decoder_num_hidden_layers']
        self.decoder = nn.ModuleDict({
            'embed_tokens': nn.Embedding(cfg['vocab_size'], cfg['hidden_size']),
            'layers': nn.ModuleList([DecoderLayer(cfg) for _ in range(dec_layers)]),
            # FIX #6: Missing decoder final LayerNorm
            'norm': nn.LayerNorm(cfg['hidden_size'], bias=False),
        })
        self.proj_out = nn.Linear(cfg['hidden_size'], cfg['vocab_size'], bias=False)
    
    def encode(self, wav):
        x = wav.unsqueeze(1)
        x = torch.tanh(self.encoder['conv1'](x))
        x = self.encoder['groupnorm'](x)
        x = F.gelu(self.encoder['conv2'](x))
        x = F.gelu(self.encoder['conv3'](x))
        x = x.transpose(1, 2)  # [B, T', dim]
        # Bidirectional (no mask needed for encoder)
        for layer in self.encoder['layers']:
            x = layer(x, mask=None)
        x = self.encoder['layer_norm'](x)
        return x
    
    def decode(self, tokens, enc):
        x = self.decoder['embed_tokens'](tokens)
        T = x.shape[1]
        # Causal mask for decoder self-attention
        self_mask = torch.triu(torch.ones(T, T, device=x.device) * float('-inf'), diagonal=1)
        # No mask for cross-attention
        cross_mask = None
        for layer in self.decoder['layers']:
            x = layer(x, enc, self_mask, cross_mask)
        # FIX #6: Apply decoder final LayerNorm before lm_head
        x = self.decoder['norm'](x)
        return self.proj_out(x)
    
    def forward(self, wav, tokens):
        enc = self.encode(wav)
        return self.decode(tokens, enc)

In [6]:
# FIX #7: Proper weight remapping from HF checkpoint keys to our scratch model keys
def remap_hf_weights(hf_state_dict, cfg):
    """Map HuggingFace checkpoint keys to scratch model keys.
    
    HF checkpoint keys have 'model.' prefix (e.g. model.encoder.conv1.weight).
    Our model uses encoder/decoder directly (e.g. encoder.conv1.weight).
    proj_out.weight is tied to decoder.embed_tokens.weight.
    """
    new_state = {}
    
    for hf_key, tensor in hf_state_dict.items():
        my_key = hf_key
        if my_key.startswith('model.'):
            my_key = my_key[len('model.'):]
        new_state[my_key] = tensor
    
    # Handle tied weights: proj_out.weight = decoder.embed_tokens.weight
    # (tie_word_embeddings=true in config, checkpoint only stores embed_tokens)
    if 'proj_out.weight' not in new_state and 'decoder.embed_tokens.weight' in new_state:
        new_state['proj_out.weight'] = new_state['decoder.embed_tokens.weight']
    
    return new_state


def load_moonshine(path):
    cfg = json.load(open(f"{path}/config.json"))
    model = Moonshine(cfg)
    hf_state = load_file(f"{path}/model.safetensors")
    
    # Remap HF keys to our model's keys
    remapped = remap_hf_weights(hf_state, cfg)
    
    # Load weights — report any mismatches
    missing, unexpected = model.load_state_dict(remapped, strict=False)
    if missing:
        print(f"WARNING: {len(missing)} missing keys: {missing[:5]}")
    if unexpected:
        print(f"WARNING: {len(unexpected)} unexpected keys: {unexpected[:5]}")
    if not missing and not unexpected:
        print("All weights loaded successfully!")
    
    tokenizer = Tokenizer.from_file(f"{path}/tokenizer.json")
    pre = json.load(open(f"{path}/preprocessor_config.json"))
    return model.eval(), tokenizer, pre


In [7]:
def transcribe(wav_path, model_dir="./moonshine-tiny"):
    import torchaudio
    model, tok, pre = load_moonshine(model_dir)
    wav, sr = torchaudio.load(wav_path)
    if sr!= 16000: wav = torchaudio.functional.resample(wav, sr, 16000)
    wav = wav.mean(0, keepdim=True) # mono
    with torch.no_grad():
        enc = model.encode(wav)
        # greedy decode
        tokens = torch.tensor([[1]]) # bos
        for _ in range(512):
            logits = model.decode(tokens, enc)[:, -1]
            nxt = logits.argmax(-1, keepdim=True)
            tokens = torch.cat([tokens, nxt], dim=1)
            if nxt.item() == 2: break # eos
    text = tok.decode(tokens[0].tolist(), skip_special_tokens=True)
    return text


In [8]:
if __name__ == "__main__":
    print(transcribe("output_16k.wav"))

All weights loaded successfully!
Recommended approach for personal projects use a pre-trained Codic in Codic sound stream, or Vocos collect small dataset one to ten hours for decent results.


In [10]:
if __name__ == "__main__":
    print(transcribe("output.wav"))

All weights loaded successfully!
Recommended approach for personal projects use a pre-trained Codic in Codic sound stream, or Vocos collect small dataset one to ten hours for decent results.


In [ ]:
from transformers import MoonshineForConditionalGeneration, AutoProcessor
import torch
import torchaudio

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = MoonshineForConditionalGeneration.from_pretrained('UsefulSensors/moonshine-tiny').to(device).to(torch_dtype)
processor = AutoProcessor.from_pretrained("UsefulSensors/moonshine-tiny")

# Load your wav file
waveform, sample_rate = torchaudio.load("output_16k.wav")

# Convert stereo → mono if needed
if waveform.shape[0] > 1:
    waveform = waveform.mean(dim=0)

# Resample if needed
target_sr = processor.feature_extractor.sampling_rate
if sample_rate != target_sr:
    resampler = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sr)
    waveform = resampler(waveform)

# Convert to numpy (processor expects numpy or list)
audio_array = waveform.squeeze().numpy()

# Process input
inputs = processor(
    audio_array,
    sampling_rate=target_sr,
    return_tensors="pt"
)

# Move to device
inputs = {k: v.to(device) for k, v in inputs.items()}

# Token limit logic
token_limit_factor = 6.5 / processor.feature_extractor.sampling_rate
seq_lens = inputs["attention_mask"].sum(dim=-1)
max_length = int((seq_lens * token_limit_factor).max().item())

# Generate
generated_ids = model.generate(**inputs, max_length=max_length)

print(processor.decode(generated_ids[0], skip_special_tokens=True))